# Lab4.4 — Avaliação Cruzada de Áudio (MUSHRA-like) — v2
**Parte A | 30–60 min.** Um grupo prepara estímulos; o **grupo parceiro** avalia sem conhecer codec e bitrate.
**MUSHRA-like didático:** referência identificada, quatro versões codificadas, referência oculta e âncora de baixa qualidade. Escala de 0–100. **Não é ensaio normativo ITU-R BS.1534.**
**G1 ↔ G2; G3 ↔ G4; G5 ↔ G6.** Execute a seção 1 no grupo produtor e a seção 2 no grupo avaliador.

In [ ]:
from pathlib import Path
import os
from google.colab import drive
drive.mount('/content/drive')
GRUPO = "G4"  # ALTERAR: G1, G2, ..., G6
RAIZ = Path('/content/drive/MyDrive/Lab4_CSM')
PASTA = RAIZ / GRUPO
PASTA.mkdir(parents=True, exist_ok=True)
print("Pasta:", PASTA)

## 1. PRODUTOR — gerar pacote anônimo
É preciso ter executado o Lab4. A pasta `pacote_audio` é destinada ao grupo avaliador; o gabarito permanece separado. Para a âncora didática usa-se um filtro passa-baixas aplicado ao WAV de referência.

In [ ]:
import subprocess, random, shutil, json, pandas as pd
from IPython.display import Audio, display
from ipywidgets import VBox, HBox, IntSlider, Button, HTML, Output
par={'G1':'G2','G2':'G1','G3':'G4','G4':'G3','G5':'G6','G6':'G5'}
referencia=PASTA/'audio'/'referencia.wav'
codificados=PASTA/'audio'/'codificados'
pacote=PASTA/'pacote_audio'; pacote.mkdir(exist_ok=True)
def ff(args):
    p=subprocess.run(args,capture_output=True,text=True)
    if p.returncode:raise RuntimeError(p.stderr[-1000:])
def preparar_audio():
    assert referencia.exists(),'Execute Lab4.1 antes.'
    # Quatro condições: dois codecs a 48k e dois a 128k.
    arquivos=[codificados/'mp3_48k.mp3',codificados/'mp3_128k.mp3',
              codificados/'aac_48k.m4a',codificados/'vorbis_128k.ogg']
    assert all(p.exists() for p in arquivos),'Arquivos codificados ausentes'
    for p in pacote.glob('S*.wav'):p.unlink()
    shutil.copy2(referencia,pacote/'referencia_identificada.wav')
    ancora=pacote/'ancora_tmp.wav'
    ff(['ffmpeg','-y','-i',str(referencia),'-af','lowpass=f=3500','-c:a','pcm_s16le',str(ancora)])
    condicoes=[('REF_OCULTA',referencia),('ANCORA',ancora)]+[(p.stem,p) for p in arquivos]
    random.SystemRandom().shuffle(condicoes)
    chave=[]
    for i,(cond,src) in enumerate(condicoes,1):
        nome=f'S{i:02d}.wav'
        ff(['ffmpeg','-y','-i',str(src),'-ar','44100','-ac','2','-c:a','pcm_s16le',str(pacote/nome)])
        chave.append({'codigo':nome,'condicao':cond,'origem':src.name})
    ancora.unlink()
    pd.DataFrame(chave).to_csv(PASTA/'audio'/'gabarito_audio_PRIVADO.csv',index=False)
    (pacote/'info.json').write_text(json.dumps({'produtor':GRUPO,'avaliador':par[GRUPO]},ensure_ascii=False))
    print('Compartilhe somente:',pacote,'com',par[GRUPO])
preparar_audio()

## 2. AVALIADOR — teste anônimo
**Altere `GRUPO_PRODUTOR` para o grupo parceiro.** Cada integrante executa individualmente, ouve a referência e os seis estímulos, e atribui notas. Use um identificador de avaliador sem nome pessoal (A1–A4). É permitido repetir a escuta.

In [ ]:
GRUPO_PRODUTOR = par[GRUPO]  # grupo cujo pacote será avaliado
AVALIADOR = 'A1'  # A1, A2, A3, A4
origem=RAIZ/GRUPO_PRODUTOR/'pacote_audio'
assert (origem/'referencia_identificada.wav').exists(),f'Pacote indisponível: {origem}'
arquivos=sorted(origem.glob('S*.wav'))
assert len(arquivos)==6,'Esperados 6 estímulos anônimos.'
display(HTML('<b>Referência identificada (não atribuir nota)</b>'))
display(Audio(filename=str(origem/'referencia_identificada.wav')))
controles=[]
for p in arquivos:
    display(HTML(f'<b>Estímulo {p.stem}</b>'))
    display(Audio(filename=str(p)))
    slider=IntSlider(value=50,min=0,max=100,step=1,description=p.stem,continuous_update=False)
    display(slider)
    controles.append((p,slider))
botao=Button(description='Salvar notas',button_style='success')
saida=Output();display(botao,saida)
def salvar(_):
    with saida:
        saida.clear_output()
        destino=PASTA/'avaliacoes_audio';destino.mkdir(exist_ok=True)
        df=pd.DataFrame([{'grupo_avaliador':GRUPO,'grupo_produtor':GRUPO_PRODUTOR,
                          'avaliador':AVALIADOR,'estimulo':p.name,'nota_0_100':s.value} for p,s in controles])
        arquivo=destino/f'{GRUPO_PRODUTOR}_{AVALIADOR}.csv'
        df.to_csv(arquivo,index=False)
        print('Salvo:',arquivo,'— compartilhe o CSV com o produtor.')
botao.on_click(salvar)

## 3. PRODUTOR — interpretação
Depois de receber os CSVs, una as notas ao gabarito privado e compare condições. Não divulgue o gabarito antes do término das avaliações.

In [ ]:
# Copie os CSVs recebidos para PASTA/'audio'/'resultados_recebidos'
recebidos=PASTA/'audio'/'resultados_recebidos'
chave=PASTA/'audio'/'gabarito_audio_PRIVADO.csv'
if chave.exists() and recebidos.exists():
    arquivos_csv=list(recebidos.glob('*.csv'))
    if arquivos_csv:
        notas=pd.concat([pd.read_csv(p) for p in arquivos_csv],ignore_index=True)
        resultado=notas.merge(pd.read_csv(chave),left_on='estimulo',right_on='codigo',how='left')
        display(resultado.groupby('condicao',dropna=False)['nota_0_100'].agg(['count','mean','std']))
    else: print('Aguardando CSVs do grupo parceiro.')
else: print('Prepare o teste e receba os CSVs antes de interpretar.')